<a href="https://colab.research.google.com/github/Neha-90/ai-mental-health-chatbot/blob/main/aimentalhealthchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 1. INSTALL
# =========================
!pip install -q transformers datasets accelerate

# =========================
# 2. IMPORTS
# =========================
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

# =========================
# 3. LOAD DATASET
# =========================
dataset = load_dataset("ShenLab/MentalChat16K")
data = dataset["train"].select(range(300))

# =========================
# 4. LOAD MODEL (FIXED)
# =========================
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

# =========================
# 5. FORMAT DATA
# =========================
def format_data(example):
    text = (
        "You are a kind mental health assistant.\n\n"
        f"User: {example['input']}\n"
        f"Assistant: {example['output']}"
    )
    return {"text": text}

data = data.map(format_data)

# =========================
# 6. TOKENIZE
# =========================
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_data = data.map(tokenize, batched=True)

tokenized_data = tokenized_data.remove_columns(
    ["instruction", "input", "output", "text"]
)

# =========================
# 7. TRAINING
# =========================
training_args = TrainingArguments(
    output_dir="./mental_chatbot",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data
)

trainer.train()

# =========================
# 8. CHAT FUNCTION
# =========================
def chat(user_input):
    device = model.device

    prompt = (
        "You are a kind mental health assistant.\n\n"
        f"User: {user_input}\nAssistant:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs["input_ids"],
        max_length=150,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Assistant:")[-1].strip()

# =========================
# 9. TEST
# =========================
print("Test 1:")
print(chat("I feel anxious"))

print("\nTest 2:")
print(chat("I am feeling very lonely these days"))

print("\nTest 3:")
print(chat("I am stressed about my exams and can't focus"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Interview_Data_6K.csv:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Synthetic_Data_10K.csv:   0%|          | 0.00/32.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16084 [00:00<?, ? examples/s]

...

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,2.728087
40,2.350956
60,2.234973
80,2.166285
100,1.969216
120,1.973274
140,1.994209
160,1.908729
180,1.904863
200,1.855176


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Test 1:
It's natural to be feeling overwhelmed and drained by things, especially when it comes time for me to focus on my work or hobbies rather than pursue personal projects that might benefit from this process [Name]. However, the desire is complex enough not only in terms of motivation but also as well.[1] This can lead to feelings of guilt towards myself due both individuals' actions and their emotions[2], which may negatively impact our overall emotional wellbeing (e-mailing). In general, we face these challenges with respectability issues such being common among people who have been experiencing chronic pain since childhood,[3][4],[5]; however there appears no significant difference between

Test 2:
Your feelings of loneliness and anxiety can be overwhelming, especially when you're around the same time as your partner or friends in bed (or at least one person). It's important to remember that there is no such thing as isolation nor attachment – both physical and emotional pain-re